In [21]:
import sys
from pathlib import Path
from difflib import get_close_matches
import scipy.io
from scipy.stats import ks_2samp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_csv('../prepared_data/story_aggregated_data.csv')

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   story_name            247 non-null    object 
 1   fool_hero_mean        247 non-null    float64
 2   fool_hero_std         247 non-null    float64
 3   angel_demon_mean      247 non-null    float64
 4   angel_demon_std       247 non-null    float64
 5   trad_adventurer_mean  247 non-null    float64
 6   trad_adventurer_std   247 non-null    float64
 7   lone_wolf_diva_mean   247 non-null    float64
 8   lone_wolf_diva_std    247 non-null    float64
 9   outcast_soph_mean     247 non-null    float64
 10  outcast_soph_std      247 non-null    float64
 11  brute_geek_mean       247 non-null    float64
 12  brute_geek_std        247 non-null    float64
 13  cast_size             247 non-null    int64  
 14  is_comedy             247 non-null    int64  
dtypes: float64(12), int64(2

In [23]:
mat_path = '../datasets/archetypometricsdata2000.mat'
mat = scipy.io.loadmat(str(mat_path), simplify_cells=True)

archetype_space = mat["data_archetype_space"]["character_component_norms"]
ds = mat["data_stories"]
story_names = ds["storyverses"]
story_char_idx = ds["storycharacterindices"]

story_index = {
    story_names[i]: (story_char_idx[i] - 1)
    for i in range(len(story_names))
}

def radius_gyration(A):
    centroid = np.mean(A, axis=1, keepdims=True)
    differences = A - centroid
    squared_distances = np.sum(differences ** 2, axis=0)
    return np.sqrt(np.mean(squared_distances))


In [24]:
gyration_vals = {}
for story_name in df['story_name']:  # or df.index if story_name is the index
    char_idx = np.atleast_1d(story_index[story_name])
    scores = archetype_space[:6, char_idx]  # (6, n_chars)
    gyration_vals[story_name] = radius_gyration(scores)

df['cast_gyration'] = df['story_name'].map(gyration_vals)  # adjust if story_name is index


In [25]:
df.head()

,story_name,fool_hero_mean,fool_hero_std,angel_demon_mean,angel_demon_std,trad_adventurer_mean,trad_adventurer_std,lone_wolf_diva_mean,lone_wolf_diva_std,outcast_soph_mean,outcast_soph_std,brute_geek_mean,brute_geek_std,cast_size,is_comedy,cast_gyration
0,(500) Days of Summer,8.353373,12.346157,-7.509349,6.439074,13.161957,19.002881,11.691623,9.962875,-11.954536,21.885571,8.717333,2.582246,2,1,33.762462
1,10 Things I Hate About You,14.097583,23.374889,9.354443,31.610384,8.895335,24.075384,11.150856,27.514599,-5.885553,12.538668,-5.884582,16.216885,6,1,57.467249
2,8 Mile,33.487735,0.000000,18.474906,0.000000,23.600837,0.000000,-23.653452,0.000000,-15.241622,0.000000,4.840814,0.000000,1,0,0.000000
3,A Series of Unfortunate Events,37.294823,31.800358,3.632574,54.001095,-2.139105,13.041584,2.745035,4.849898,-10.481052,8.611981,16.124795,5.170841,3,1,64.976067
4,A Star Is Born,16.504863,13.921006,1.235229,21.970246,27.318987,3.920331,-7.981630,5.684261,-11.096272,7.850583,3.498290,5.756921,2,0,28.617123


In [26]:
# get df['cast_gyration'].value_counts() greater than 0 should be 239
df[df['cast_gyration'] > 0]['cast_gyration'].value_counts()

cast_gyration
33.762462    1
57.467249    1
64.976067    1
28.617123    1
29.358794    1
            ..
44.200316    1
39.663632    1
48.142242    1
41.947301    1
48.754572    1
Name: count, Length: 239, dtype: int64

In [27]:
comedy = df[df['is_comedy'] == 1]
drama  = df[df['is_comedy'] == 0]

feature_cols = [c for c in df.columns if c not in ('is_comedy', 'story_name')]
results = []

for col in feature_cols:
    stat, pval = ks_2samp(comedy[col].values, drama[col].values)
    results.append({'feature': col, 'ks_stat': stat, 'p_value': pval})

ks_df = (pd.DataFrame(results)
           .sort_values('ks_stat', ascending=False)
           .reset_index(drop=True))

ks_df['significant'] = ks_df['p_value'] < 0.05
ks_df.round(4)


,feature,ks_stat,p_value,significant
0,trad_adventurer_mean,0.3554,0.0000,True
1,fool_hero_mean,0.2985,0.0001,True
2,lone_wolf_diva_mean,0.2759,0.0003,True
3,cast_gyration,0.2578,0.0009,True
4,outcast_soph_std,0.1783,0.0476,True
5,angel_demon_mean,0.1761,0.0520,False
6,trad_adventurer_std,0.1600,0.0976,False
7,angel_demon_std,0.1448,0.1666,False
8,brute_geek_mean,0.1070,0.4954,False
9,fool_hero_std,0.1037,0.5355,False


In [32]:
# define features and target
target = 'is_comedy'
exclude = {'is_comedy', 'story_name'}

baseline_features = [c for c in df.columns if c not in exclude and c != 'cast_gyration']
gyration_features = [c for c in df.columns if c not in exclude]

std_features = [c for c in df.columns if c.endswith('_std')]
no_std_features = [c for c in gyration_features if c not in std_features]

X_baseline = df[baseline_features].values
X_gyration  = df[gyration_features].values
X_no_std    = df[no_std_features].values
y = df[target].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000))
    ])
}

scoring = ['roc_auc', 'f1', 'accuracy']
results = []

for model_name, model in models.items():
    for feat_name, X in [
        ('baseline', X_baseline),
        ('+ gyration', X_gyration),
        ('gyration replaces std', X_no_std),
    ]:
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        results.append({
            'model': model_name,
            'features': feat_name,
            'AUC':      scores['test_roc_auc'].mean(),
            'AUC_std':  scores['test_roc_auc'].std(),
            'F1':       scores['test_f1'].mean(),
            'F1_std':   scores['test_f1'].std(),
            'Accuracy': scores['test_accuracy'].mean(),
            'Acc_std':  scores['test_accuracy'].std(),
        })

results_df = pd.DataFrame(results)
results_df.round(3)


,model,features,AUC,AUC_std,F1,F1_std,Accuracy,Acc_std
0,Logistic Regression,baseline,0.724,0.037,0.492,0.078,0.688,0.036
1,Logistic Regression,+ gyration,0.726,0.037,0.502,0.055,0.696,0.035
2,Logistic Regression,gyration replaces std,0.746,0.024,0.543,0.077,0.716,0.039


In [30]:
df[gyration_features].corr()[['cast_gyration']].round(3).sort_values('cast_gyration', ascending=False)

,cast_gyration
cast_gyration,1.000
angel_demon_std,0.762
trad_adventurer_std,0.673
fool_hero_std,0.656
outcast_soph_std,0.595
lone_wolf_diva_std,0.514
brute_geek_std,0.465
cast_size,0.443
lone_wolf_diva_mean,0.306
angel_demon_mean,0.191
